# Stage 1: Plastome Ingestion, Biological Quality Control & Structural De-Isomerization

## 1. Scientific Overview & Problem Context
Chloroplast genomes (plastomes) in monocots, particularly within the family Musaceae (*Musa*, *Ensete*, and *Musella*), possess a canonical circular quadripartite architecture (~155–170 kb):
- **Large Single Copy (LSC):** Hosts key photosystem and carbon fixation core complexes.
- **Small Single Copy (SSC):** Enriched in NADH dehydrogenase genes with elevated substitution rates.
- **Inverted Repeats ($\text{IR}_\text{A}$ & $\text{IR}_\text{B}$):** Identical copies in reverse orientation maintained by copy-dependent gene conversion.

### Computational Pitfalls in Public Sequence Repositories
GenBank plastome assemblies cannot be aligned naively due to:
1. **Arbitrary Linearization:** Deposited FASTA files start at inconsistent assembly break points.
2. **Structural Heteroplasmy (SSC Flip-Flop):** Chloroplast DNA exists in two equimolar structural isomers inside a plant cell because of homologous recombination between the IRs. Assemblers pick an SSC orientation arbitrarily, causing false 180° chromosomal inversion artifacts.
3. **Mitochondrial & NUPT Contamination:** Mitochondrial fragments (~15 kb) and Nuclear Plastid DNA (NUPTs) are frequently mislabeled in public databases.

This notebook automates data curation, strict biological quality control (length and GC-content boundaries), exact mathematical detection of Inverted Repeats, and structural phase standardization.

In [ ]:
# Cell 1: Environment Detection & Automated Dependency Bootstrap
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("[Environment] Google Colab detected. Initializing workspace...")
    !git clone https://github.com/satuankecepatan/musa-hypothetical-structure.git /content/musa-hypothetical-structure
    %cd /content/musa-hypothetical-structure
    !pip install -q -r requirements.txt
    !apt-get update -qq && apt-get install -y -qq iqtree mafft
else:
    print("[Environment] Running in local workstation environment.")
    # Ensure project root is in sys.path
    root_path = Path.cwd().resolve()
    if str(root_path) not in sys.path:
        sys.path.insert(0, str(root_path))

import scripts.config as config
from scripts.tool_resolver import get_tool_status

print("Project Root:", config.BASE_DIR)
tool_status = get_tool_status()
for tool, path in tool_status.items():
    print(f"  {tool:<10}: {path or 'Not Resolved'}")

## 2. Taxon Sampling & Biological QC Curation
We sample high-confidence wild diploid representatives across all 4 recognized *Musa* sections (*Eumusa*, *Rhodochlamys*, *Callimusa*, and *Australimusa*) plus rooted outgroup genera (*Ensete* and *Musella*).

Every genome must satisfy:
- **Canonical Length:** $130\text{--}185\text{ kb}$
- **Plastid GC-Content:** $36.0\% \le \text{GC} \le 38.0\%$ (filtering out NUPT drifts and mitochondrial contaminants)
- **Intact Photosystem Markers:** Preserving intact CDS for $rbcL$, $matK$, and $psbA$.

In [ ]:
# Cell 2: Automated Entrez Download & Biological QC
import json
import pandas as pd
from scripts.curation import curate_tier_1

metadata_list = curate_tier_1()
df_meta = pd.DataFrame([m.__dict__ for m in metadata_list])
display(df_meta[["accession", "taxon", "section", "length", "gc_percent", "qc_reason"]])

## 3. Exact Mathematical Detection of Inverted Repeats
Rather than relying on inconsistent GenBank `repeat_region` tags, we apply a vectorized seed-and-extend reverse-complement search to locate the exact base coordinates of $\text{IR}_\text{B}$ and $\text{IR}_\text{A}$, segmenting each genome into $[LSC, IR_B, SSC, IR_A]$.

In [ ]:
# Cell 3: Compute Quadripartite Boundaries & Flanking Junction Genes
from scripts.ir_detector import process_all_curated_junctions

junctions = process_all_curated_junctions()
df_junc = pd.DataFrame([j.__dict__ for j in junctions])
display(df_junc[["accession", "genome_length", "lsc_len", "irb_len", "ssc_len", "ira_len"]])

## 4. Circular Phase Standardization & SSC Flip-Flop Resolution
We standardize all accessions to start at the $5'$ boundary of the LSC upstream of $trnH\text{-GUG}$. We then inspect the relative synteny of anchor gene $ndhF$. If an accession is an inverted isomer, we reverse-complement its SSC partition to ensure 100% collinearity.

In [ ]:
# Cell 4: Normalize Structural Phase & Export Multi-FASTAs
from scripts.normalizer import normalize_dataset

norm_summary = normalize_dataset()
df_norm = pd.DataFrame.from_dict(norm_summary, orient="index")
display(df_norm)

## 5. Visualizing Inverted Repeat Expansion/Contraction Across Sections
We generate a publication-grade synteny diagram showing the quadripartite compartment lengths across the *Musa* sections and Musaceae outgroups.

In [ ]:
# Cell 5: Publication Figure Export (IR Junction Dynamics)
from IPython.display import Image, display
from scripts.visualization import plot_junction_shifts
import scripts.config as config

junc_json = config.DATA_DIR / "tier1_ir_junctions.json"
fig_prefix = config.OUTPUT_DIR / "figure_junction_shifts"
fig_path = plot_junction_shifts(junc_json, fig_prefix)

display(Image(filename=str(fig_path)))